<div style="width: 100%; text-align: center;">
    <div style="background-color:#007F00; padding: 0.5rem;">
        <h1 style="font-weight: bold; font-size: 2.5em; color: black;"> AGRHYMET CENTRE CLIMATIQUE REGIONAL POUR L'AFRIQUE DE L'OUEST ET LE SAHEL</h1>
   </div>

   <div style="text-align: center;">
  <img src="https://www.sareco.org/wp-content/uploads/2017/07/plrDvYX1.jpg" width="200">
</div>

<a id="1"></a>
### <p style="padding:10px;background-color:#000000 ;margin:0;color:#007F00;font-family:#newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500"> Prevision GloFAS et carte de risque d'inondation </p>

## Objectif

À partir de la prévision GloFAS du jour (jusqu'à 30 jours d'échéance, 51
membres), produire une **carte de risque d'inondation** par point d'intérêt
et par échéance : *normal*, *risque faible*, *risque modéré* ou *risque
sévère*.

À défaut de seuils de risque officiels (périodes de retour 2/5/20 ans,
calées par ajustement statistique — voir `glofas_risk.py`), on utilise des
**quantiles empiriques calculés sur l'historique** de chaque point comme
seuils de référence : une approximation simple mais transparente, adaptée à
la démonstration.

**Prérequis** : avoir déjà exécuté `download_glofas_data.ipynb` (téléchargement
+ extraction de l'historique aux points d'intérêt) — ce notebook réutilise
`resultats/extraction_points.csv` et `resultats/extraction_series.csv`
produits là-bas. La maille retenue pour chaque point (`lon_pixel`/`lat_pixel`)
est réutilisée telle quelle pour la prévision, afin de comparer la prévision
aux seuils historiques sur exactement la même maille.

In [ ]:
import sys, os
from pathlib import Path
from glofas_forecast import download_glofas_forecast, extract_glofas_forecast_at_points
from glofas_risk import compute_historical_thresholds, classify_forecast_risk
from glofas_visualize import build_risk_map

## 1. Configuration EWDS

Comme pour le téléchargement de l'historique.

In [ ]:
ewds_config = Path.home() / ".cdsapirc-ewds"

if not ewds_config.exists():
    raise FileNotFoundError(f"Configuration EWDS introuvable : {ewds_config}")

os.environ["CDSAPI_RC"] = str(ewds_config)
print("Configuration sélectionnée :", ewds_config)

## 2. Télécharger la prévision du jour

Un fichier par type de membre (`control_forecast` = 1 membre,
`ensemble_perturbed_forecasts` = 50 membres) et par date d'émission,
horizon 30 jours par défaut. Comme pour l'historique, restreignez `area`
à votre zone d'intérêt — une requête de prévision couvre déjà jusqu'à 30
échéances (et 50 membres pour les perturbés), donc peut vite devenir
volumineuse sur une zone globale.

Pour une simple démonstration (plus rapide), limitez `product_types` à
`("control_forecast",)` : un seul membre, donc pas de probabilité de
dépassement calculable, mais un aperçu immédiat.

In [ ]:
from datetime import date

resultats_dl = download_glofas_forecast(
    issue_date=date.today().isoformat(),  # ou une date passée, ex. "2026-09-01"
    max_days=15,                           # horizon en jours (30 = horizon complet de cems-glofas-forecast)
    area=(7.0, 8.0, 2.0, 16.0),             # nord, ouest, sud, est -- à adapter à votre zone
    product_types=("control_forecast", "ensemble_perturbed_forecasts"),
    output_dir="glofas_forecast_data",
)
resultats_dl

## 3. Extraire la prévision aux points d'intérêt

Réutilise la maille déjà déterminée par l'extraction historique
(`resultats/extraction_points.csv`) : **pas** de nouveau recalage sur la
prévision (voir la note en tête de `glofas_forecast.py` pour la raison).

In [ ]:
prevision = extract_glofas_forecast_at_points(
    points_meta="resultats/extraction_points.csv",
    forecast_dir="glofas_forecast_data",
    output="resultats/prevision_series.csv",
)
prevision.head()

## 4. Calculer les seuils de risque (quantiles historiques)

`basis="daily"` (défaut) : quantile sur toutes les valeurs journalières —
simple, mélange saison sèche et saison des pluies. `basis="annual_max"` :
quantile sur les maxima annuels, plus proche de la notion de période de
retour, mais demande un historique plus long (5 ans minimum, idéalement
bien plus) pour rester fiable — un avertissement s'affiche sinon.

In [ ]:
seuils = compute_historical_thresholds(
    "resultats/extraction_series.csv",
    basis="daily",                 # ou "annual_max"
    quantiles={"seuil_faible": 0.80, "seuil_modere": 0.90, "seuil_severe": 0.98},
    output="resultats/seuils_risque.csv",
)
seuils

## 5. Classer la prévision par rapport aux seuils

Pour chaque point et chaque échéance : statistique centrale de l'ensemble
(médiane par défaut, robuste aux membres extrêmes), étendue de l'ensemble,
valeur du membre de contrôle, probabilité de dépassement de chaque seuil
(fraction des 51 membres au-delà), et catégorie de risque.

In [ ]:
risque = classify_forecast_risk(
    "resultats/prevision_series.csv",
    seuils,
    central_stat="median",
    output="resultats/prevision_risque.csv",
)
risque[["id", "issue_date", "leadtime_hours", "membre_central", "minimum", "maximum", "risque"]]

## 6. Carte de risque

Une échéance à la fois (`build_risk_map`), ou un sélecteur interactif
d'échéance et de date d'émission (`interactive_risk_explorer`, comme pour
les séries temporelles historiques).

In [ ]:
carte = build_risk_map(
    risque, "resultats/extraction_points.csv",
    output_html="resultats/carte_risque.html",
    leadtime_hours=int(risque["leadtime_hours"].min()),  # première échéance disponible ; ajustez au besoin
)
carte

### Sélecteur interactif (échéance + date d'émission)

Nécessite `ipywidgets` (déjà inclus dans l'environnement de l'atelier).

In [ ]:
from glofas_visualize import interactive_risk_explorer

interactive_risk_explorer(risque, "resultats/extraction_points.csv")

---

**Limites à garder en tête pour l'atelier** : les seuils sont des quantiles
empiriques de l'historique disponible, pas des périodes de retour ajustées
statistiquement (méthode utilisée en opérationnel par GloFAS) -- ils sont
d'autant plus fiables que l'historique extrait est long. La catégorie de
risque affichée sur la carte repose sur la statistique centrale de
l'ensemble (médiane) ; gardez un œil sur les colonnes
`probabilite_depassement_*` de `resultats/prevision_risque.csv` pour une
lecture probabiliste complète (ex. « 70 % des membres dépassent le seuil
sévère à J+5 »).